Ollama client version is `0.2.1`

What's the content of the file related to gemma?

`2b`

In [5]:
10 * 10

100

1.7

gemma:2b 

In [9]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [25]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch('http://localhost:9200')

In [26]:
es_client

<Elasticsearch(['http://localhost:9200'])>

In [27]:
index_settings = {
    "settings": {
        "number_of_shards":1,
        "number_of_replicas":0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"}
        }
    }
}

index_name = "course-questions"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [28]:
from tqdm.auto import tqdm 

In [29]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [35]:
query = "What's the formula for energy?"

search_query = {
    'size': 5,
    'query':{
        'bool': {
            'must': {
                'multi_match': {
                    'query': query,
                    'fields': ['question^4', 'text'],
                    'type': 'best_fields'
                }
            },
        }
    }
}
search_results = es_client.search(index=index_name, body=search_query)

In [36]:
search_results['hits']['hits'][0]

{'_index': 'course-questions',
 '_id': '5AntlpABtTdWgiA_CGF8',
 '_score': 39.57298,
 '_source': {'text': 'In Question 7 we are asked to calculate\nThe initial problem  can be solved by this, where a Matrix X is multiplied by some unknown weights w resulting in the target y.\nAdditional reading and videos:\nOrdinary least squares\nMultiple Linear Regression in Matrix Form\nPseudoinverse Solution to OLS\nAdded by Sylvia Schmitt\nwith commends from Dmytro Durach',
  'section': '1. Introduction to Machine Learning',
  'question': 'Question 7: Mathematical formula for linear regression',
  'course': 'machine-learning-zoomcamp'}}

In [37]:
query = "What's the formula for energy?"

search_query = {
    "size": 3,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    }
}

search_results = es_client.search(index=index_name, body=search_query)

In [38]:
search_results['hits']['hits'][2]

{'_index': 'course-questions',
 '_id': '0AntlpABtTdWgiA_BmGi',
 '_score': 17.107508,
 '_source': {'text': "Leaderboard Links:\n2023 - https://docs.google.com/spreadsheets/d/e/2PACX-1vSNK_yGtELX1RJK1SSRl4xiUbD0XZMYS6uwHnybc7Mql-WMnMgO7hHSu59w-1cE7FeFZjkopbh684UE/pubhtml\n2022 - https://docs.google.com/spreadsheets/d/e/2PACX-1vQzLGpva63gb2rIilFnpZMRSb-buyr5oGh8jmDtIb8DANo4n6hDalra_WRCl4EZwO1JvaC4UIS62n5h/pubhtml\nPython Code:\nfrom hashlib import sha1\ndef compute_hash(email):\nreturn sha1(email.lower().encode('utf-8')).hexdigest()\nYou need to call the function as follows:\nprint(compute_hash('YOUR_EMAIL_HERE'))\nThe quotes are required to denote that your email is a string.\n(By Wesley Barreto)\nYou can also use this website directly by entering your email: http://www.sha1-online.com. Then, you just have to copy and paste your hashed email in the “research” bar of the leaderboard to get your scores.\n(Mélanie Fouesnard)",
  'section': '1. Introduction to Machine Learning',
  'question'

In [50]:
context_template = """
Q: {question}
A: {text}
""".strip()

prompt_template = """
What's the formula for energy?

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [51]:
context_pieces = []

for hit in search_results['hits']['hits']:
    doc = hit['_source']
    context_piece = context_template.format(**doc)
    context_pieces.append(context_piece)

context = '\n\n'.join(context_pieces)

In [52]:
prompt = prompt_template.format(question=query, context=context)

In [53]:
len(prompt)

1440

In [54]:
import tiktoken

In [55]:
print(prompt[:100])

What's the formula for energy?

QUESTION: What's the formula for energy?

CONTEXT:
Q: Question 7: Ma


In [49]:
encoding = tiktoken.encoding_for_model("gpt-4o")

In [56]:
len(encoding.encode(prompt))

410

In [57]:
tokens = encoding.encode(prompt)[:10]
tokens

[45350, 290, 20690, 395, 5954, 1715, 107036, 25, 51662, 290]

In [58]:
encoding.decode_single_token_bytes(tokens[5])

b'?\n\n'